# 5. Subarray Sum Equals K
**Difficulty:** 🟡 Medium · **Topic:** Arrays / Hash Maps · **Pattern:** Prefix sum + running hash map of counts (LeetCode 560)

> **DevRev context:** you have a timeline of events — each carrying a number (a ticket's handling cost, an SLA-timer tick, a usage-metric delta). You want to count how many **contiguous spans** of that timeline sum to exactly a target `k` (e.g. "how many back-to-back work windows cost exactly one SLA unit"). Scanning every span is quadratic; a **prefix sum with a hash map** answers it in one pass.

## 💡 Concepts

**Core concept(s):** **Prefix sum** + a **hash map** that counts how many times each running total has been seen.

**Why it applies here:** The sum of the span `(i, j]` equals `prefix[j] - prefix[i]`. So a span ending at `j` sums to `k` exactly when some earlier prefix equals `prefix[j] - k`. Instead of searching for that earlier prefix each time (slow), we keep a **running tally of every prefix we've seen** in a hash map and look it up in O(1).

**Key intuition:** *"How many earlier points had a running total of `current_total - k`?"* — each one marks the start of a valid span ending here.

---

### 📚 What is a prefix sum?
A **prefix sum** at index `j` is the total of all elements from the start up to `j`. Once you have running totals, the sum of any span becomes a simple **subtraction** of two of them — no re-adding. That turns an O(n) inner sum into O(1).

### 📚 What is a hash map (dict) here?
A **hash map** stores `key -> value` with average **O(1)** insert and lookup. Here the key is a *running total* and the value is *how many times we've reached that total so far*. Looking up "have I seen total `X` before, and how often?" is instant instead of a scan.

---

**Prerequisite knowledge:**
- Running/prefix sums.
- The identity `sum(i..j] = prefix[j] - prefix[i]`.
- Seeding the map with `{0: 1}` so a prefix that *itself* equals `k` is counted (empty prefix before the array).

## 📝 Problem

Given an integer array `nums` and an integer `k`, return the **number of contiguous subarrays** whose elements sum to exactly `k`. Values may be **negative**, so a sliding window does *not* work (shrinking/growing on a sum is only monotonic for non-negative values).

**Example**
```
nums = [1, 2, 3], k = 3   ->  2      # [1,2] and [3]
nums = [1, 1, 1], k = 2   ->  2      # [1,1] (positions 0-1) and [1,1] (positions 1-2)
nums = [1, -1, 0], k = 0  ->  3      # [1,-1], [0], [1,-1,0]
```

> Two approaches: a naive **check-every-subarray** `O(n^2)` and a **prefix-sum + hash map** `O(n)`.

### Approach 1 — Check Every Subarray (worst)

Fix each start index, extend the end one step at a time keeping a **running sum**, and count each time it equals `k`. That's `O(n^2)` spans — correct, but it re-walks the tail for every start.

In [ ]:
from typing import List

def subarray_sum_naive(nums: List[int], k: int) -> int:
    n = len(nums)
    count = 0
    for start in range(n):                 # every possible span start
        running = 0
        for end in range(start, n):        # extend the span one element at a time
            running += nums[end]           # O(1) update instead of re-summing the slice
            if running == k:               # this span (start..end) hits the target
                count += 1
    return count                           # total number of matching spans

### Approach 2 — Prefix Sum + Hash Map (optimal)

Walk once, keeping a **running total**. For each position ask: *how many earlier prefixes equalled `running - k`?* Each such earlier prefix is the start of a span ending here that sums to `k`. A hash map of prefix-counts makes that lookup O(1).

In [ ]:
from typing import List
from collections import defaultdict

def subarray_sum_fast(nums: List[int], k: int) -> int:
    count = 0
    running = 0
    seen = defaultdict(int)                 # running-total -> how many times we've hit it
    seen[0] = 1                             # empty prefix: lets a span from index 0 count
    for x in nums:
        running += x                        # prefix sum up to and including x
        # a span ending here sums to k iff some earlier prefix == running - k
        count += seen[running - k]          # add the number of such earlier starts (O(1))
        seen[running] += 1                  # record this prefix for future positions
    return count

In [ ]:
# Correctness check
cases = [
    ([1, 2, 3], 3, 2),
    ([1, 1, 1], 2, 2),
    ([1, -1, 0], 0, 3),
    ([3, 4, 7, 2, -3, 1, 4, 2], 7, 4),
    ([], 0, 0),
    ([0, 0, 0], 0, 6),        # every one of the 6 non-empty spans sums to 0
]
for nums, k, expected in cases:
    a = subarray_sum_naive(nums, k)
    b = subarray_sum_fast(nums, k)
    assert a == b == expected, f"mismatch on {nums}, k={k}: naive={a} fast={b} exp={expected}"
print("All tests passed")

## ⏱️ Empirically Checking the Complexities

We time each approach on inputs of growing size `n` and read the **doubling ratio**.

| Theoretical | Ratio `n`→`2n` |
|---|---|
| `O(n)`       | ≈ **2×** |
| `O(n log n)` | ≈ **2×** (slightly more) |
| `O(n^2)`     | ≈ **4×** |

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")): break
    _root = os.path.dirname(_root)
if _root not in sys.path: sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    # All zeros with k=0: EVERY span matches, so neither approach can early-exit.
    # Values are 0 -> integer arithmetic stays O(1) (no big-int blow-up), so the
    # timing reflects the number of spans visited, not the cost of adding numbers.
    return ([0] * n, 0)

solutions = {
    "check-all  O(n^2)": subarray_sum_naive,
    "prefix+map O(n)  ": subarray_sum_fast,
}
sizes = [200, 400, 800, 1600]

benchmark(solutions, make_worst_case, sizes, plot=True)

## 🧩 Patterns Learned

- **Prefix sum turns a span-sum into a subtraction:** `sum(i..j] = prefix[j] - prefix[i]` — precompute running totals so any span is O(1).
- **Hash map of prefixes = "have I seen this total before?":** counting earlier prefixes equal to `running - k` collapses an O(n^2) search into O(n).
- **Seed `{0: 1}`:** the empty prefix lets a span starting at index 0 be counted — a classic off-by-one trap if omitted.
- **Negatives kill the sliding window:** with negative values the running sum isn't monotonic, so the two-pointer/window trick fails — reach for prefix + hash map instead.
- **Signal:** "count/if any **contiguous** subarray sums to a target", especially with negatives or zeros in the data.
- **DevRev / related:** counting contiguous work-windows that hit a cost/SLA target; Two Sum (same "have I seen `target - x`?" idea, on values instead of prefixes); Subarray Sum Divisible by K; Continuous Subarray Sum.
- **Common pitfalls:** (1) forgetting `seen[0] = 1`; (2) reaching for a sliding window when values can be negative; (3) updating the map *before* the lookup (must count earlier prefixes, then record the current one).